In [1]:
import pandas as pd
from IPython.display import display 

In [2]:
file_path = '/Users/albertomarques/Proyectos/Bootcamp/Curso2026/Semana_2/Proyecto_Semana_2/Proyecto-Shark-Attacks/DateSet_Ataque_Tiburones.xls'
shark_df = pd.read_excel('/Users/albertomarques/Proyectos/Bootcamp/Curso2026/Semana_2/Proyecto_Semana_2/Proyecto-Shark-Attacks/DateSet_Ataque_Tiburones.xls')

In [3]:
shark_df.shape

(7087, 23)

In [4]:
shark_df.head()

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,...,Species,Source,pdf,href formula,href,Case Number,Case Number.1,original order,Unnamed: 21,Unnamed: 22
0,14/04,2026.0,UNprovoked,Maldives,Gaafu Alif Atoll,Kooddoo,Swimming,Not stated - on honeymoon,M,?,...,Unknown,The U.S. Sun: Simon De Marchi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3rd April,2026.0,Unprovoked,Australia,South Australia,Middleton Beach Fleurieu Peninsula Adelaide,Surfing,Oliver Tokic-Bensley,M,16,...,Bronze Whaler,ABC News: The Guardian: Andrew Currie and Bob...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,26th March,2026.0,Unprovoked,Bahamas,Andros Island,Fresh Creek,Swimming,Australian woman,F,22,...,Unknown,WIC News: Melissa Michaelson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,25th March,2026.0,Unprovoked,Australia,South Australia,Cape Jaffa Limestone Coast,Diving,Luke Kuhn,M,?,...,Great White Shark 3.5m (11.5ft),Bob Myatt GSAF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,22nd-23rd March,2026.0,Unprovoked,Australia,NSW,Little Avalon Beach,Surfing,Unknown,M,30+,...,Unknown,Melissa Michaelson: Instagram,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
''' 
We normalize the names of the columns in the dataset and keep the colums with valuable infomration to our proyect.
'''
shark_df.columns = shark_df.columns.str.lower().str.strip().str.replace(' ','_')
cols_to_keep = ['date', 'year', 'country', 'state', 'location', 'species']
shark_df = shark_df[cols_to_keep].copy()
shark_df.head()

,date,year,country,state,location,species
0,14/04,2026,maldives,Gaafu Alif Atoll,Kooddoo,unknown
1,3rd April,2026,australia,South Australia,Middleton Beach Fleurieu Peninsula Adelaide,unknown
2,26th March,2026,bahamas,Andros Island,Fresh Creek,unknown
3,25th March,2026,australia,South Australia,Cape Jaffa Limestone Coast,great white shark
4,22nd-23rd March,2026,australia,NSW,Little Avalon Beach,unknown


In [19]:
'''
Since the dataset covers an extensive period, we have decided to keep only the records from the last 26 years.
(roughly the average lifespan of a shark)
'''

def filter_years(df):
    """
    Filters the dataframe to keep only rows where the year 
    is between 2000 and 2026 (inclusive).
    """
    # We define the condition: year must be >= 2021 AND <= 2026
    df = df[(df['year'] >= 2000) & (df['year'] <= 2026)]
    
    # It is a good practice to reset the index after filtering rows
    return df.reset_index(drop=True)
# Apply the function
shark_df = filter_years(shark_df)


In [20]:
shark_df['year'] = shark_df['year'].astype('Int64') 
shark_df.head()

,date,year,country,state,location,species
0,14/04,2026,maldives,Gaafu Alif Atoll,Kooddoo,unknown
1,3rd April,2026,australia,South Australia,Middleton Beach Fleurieu Peninsula Adelaide,unknown
2,26th March,2026,bahamas,Andros Island,Fresh Creek,unknown
3,25th March,2026,australia,South Australia,Cape Jaffa Limestone Coast,great white shark
4,22nd-23rd March,2026,australia,NSW,Little Avalon Beach,unknown


In [21]:
'''
knowing the exact location where the atack was reported it's importan to our implementation.
Wee only want to keep data corresponding to column country if we can to track them to an specific state and location.
'''
shark_df['country'] = shark_df['country'].str.lower().str.strip()
shark_df = shark_df[shark_df['state'].notnull() & shark_df['location'].notnull()]

In [22]:
shark_df.shape

(2649, 6)

In [23]:
shark_df.isnull().sum()

date        0
year        0
country     0
state       0
location    0
species     0
dtype: int64

In [26]:
shark_df['species'] = shark_df['species'].fillna('unknown').str.lower().str.strip()
contains_shark = shark_df['species'].str.contains('shark', na = False)
contains_question = shark_df['species'].str.contains(r'\?|female|not|involvement|or|possibly|juvenile', na = False)
shark_df.loc[~contains_shark | contains_question , 'species'] = 'unknown'
regex_limpieza = r'\d+\s*[m\']|[0-9,\'\.\[\]\"\(\)]|\bto\b|\ba\s+|\bfemale\b|\bmale\b|\bft\b'
regex_estetica = r'\s+'
shark_df['species'] = (shark_df['species']
    .str.replace(regex_limpieza, '', regex=True)
    .str.replace(regex_estetica, ' ', regex=True)
    .str.strip())
# cleaned_species = {'shark':'unknown', 'shark involvement not confirmed':'unknown','no shark involvement':'unknown',
#                    'shark involvement prior death was not confirmed':'unknown','small shark':'unknown',
#                    'blacktip or spinner shark':'unknown'}
# shark_df['species'] = shark_df['species'].replace(cleaned_species)
shark_df['species'].value_counts().head(20)

species
unknown                   1332
shark                      334
white shark                257
bull shark                 129
tiger shark                117
blacktip shark              39
bronze whaler shark         28
nurse shark                 27
small shark                 23
wobbegong shark             21
mako shark                  18
lemon shark                 17
raggedtooth shark           16
sandtiger shark             16
reef shark                  13
oceanic whitetip shark      13
great white shark           10
blue shark                  10
grey reef shark             10
spinner shark               10
Name: count, dtype: int64

In [12]:
shark_df['country'].value_counts().head(20)

country
usa                 1341
australia            576
south africa         152
brazil                64
new zealand           60
bahamas               55
mexico                37
new caledonia         31
french polynesia      30
egypt                 28
reunion               20
spain                 19
mozambique            11
fiji                  11
philippines           10
thailand              10
cuba                   9
papua new guinea       9
indonesia              8
ecuador                8
Name: count, dtype: int64

In [13]:
shark_df['state'].value_counts().head(30)

state
Florida                  691
New South Wales          190
Hawaii                   174
California               149
Western Australia        146
Queensland               109
South Carolina            94
North Carolina            78
Western Cape Province     57
Eastern Cape Province     53
South Australia           46
Pernambuco                45
Texas                     40
Victoria                  36
KwaZulu-Natal             34
North Island              30
South Island              18
New York                  18
Abaco Islands             18
NSW                       16
Oregon                    16
Alabama                   11
Tasmania                  10
Massachusetts              9
South Province             9
Quintana Roo               8
New Jersey                 8
North Province             8
Binh Dinh Province         8
Inhambane Province         7
Name: count, dtype: int64

In [14]:
shark_df['location'].value_counts().head(30)
                                        

location
New Smyrna Beach, Volusia County                                 150
Ponce Inlet, Volusia County                                       22
Cocoa Beach, Brevard  County                                      18
Myrtle Beach, Horry County                                        17
Melbourne Beach, Brevard County                                   15
Daytona Beach, Volusia County                                     12
Isle of Palms, Charleston County                                  11
Jacksonville Beach, Duval County                                  11
New Smyrna Beach                                                  10
Ponce Inlet, New Smyrna Beach, Volusia County                      9
Daytona Beach Shores, Volusia County                               8
Quy Nhon                                                           8
Florida Keys, Monroe County                                        7
Vero Beach, Indian River County                                    7
Ormond Beach, Volusia Cou